In [ ]:
import pandas as pd

domains = ["ARC", "BAC"]
hierarchies = ["P", "C", "O", "F", "G"]

for domain in domains:
    xlsx_path = f"final/Dat_{domain}.xlsx"
    for hierarchy in hierarchies:
        var_name = f"{domain}_{hierarchy}"
        globals()[var_name] = pd.read_excel(xlsx_path, sheet_name=f"{hierarchy}_rank(%)")

print([f"{d}_{h}" for d in domains for h in hierarchies])


In [ ]:
chem_xlsx_path = "final/Dat_chem.xlsx"

SEASONS = ["S", "F", "W", "Sp"]


def load_chem_block(xlsx_path, sheet_name, usecols, n_leading_singles=0):
    """Load a Chem_*.Avg sheet: columns repeat each parameter 4x (one per
    season) with a blank spacer column between parameter groups. Renames
    using the sheet's own header text + season suffix, positionally chunked
    into groups of 4 (not by matching header text) since at least one group
    (TS/VS in Chem_Si.Avg) has a mislabeled 4th header cell in the source
    file."""
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, usecols=usecols)
    header = raw.iloc[0]
    data = raw.iloc[1:].reset_index(drop=True)

    keep = header.notna().to_numpy()
    header = header[keep].reset_index(drop=True)
    data = data.loc[:, keep]
    data.columns = range(data.shape[1])

    index_name = str(header.iloc[0]).strip()
    df = data.iloc[:, 1:].copy()
    df.index = data.iloc[:, 0].values
    df.index.name = index_name
    header = header.iloc[1:].reset_index(drop=True)

    new_columns = []
    col_pos = 0
    for _ in range(n_leading_singles):
        new_columns.append(str(header.iloc[col_pos]).strip())
        col_pos += 1

    remaining = len(header) - col_pos
    assert remaining % 4 == 0, f"expected groups of 4 seasons, got {remaining} leftover columns"
    for _ in range(remaining // 4):
        param = str(header.iloc[col_pos]).strip().replace(",", "_").replace("-", "_")
        for season in SEASONS:
            new_columns.append(f"{param}_{season}")
            col_pos += 1

    df.columns = new_columns
    return df


Sub = load_chem_block(chem_xlsx_path, "Chem_Si.Avg", usecols="C:DI", n_leading_singles=1)
M = load_chem_block(chem_xlsx_path, "Chem_Se.Avg", usecols="C:EB", n_leading_singles=0)

print("Sub:", Sub.shape, Sub.index.name)
print("M:", M.shape, M.index.name)


In [ ]:
Sub = Sub.apply(pd.to_numeric, errors="coerce")
M = M.apply(pd.to_numeric, errors="coerce")

print("Sub non-null count:", Sub.notna().sum().sum(), "/", Sub.size)
print("M non-null count:", M.notna().sum().sum(), "/", M.size)


In [ ]:
import numpy as np

# Genus-level taxa (finest rank loaded above) vs. Sub/M chemistry, Spearman
# (rank-based, robust to the compositional/non-normal abundance data).
TAXON_LEVEL = "G"
MIN_PAIRED_SAMPLES = 15  # drop correlations backed by too few overlapping samples

meta = pd.read_csv("final/metadata.csv").set_index("SampleID")
ROUND_TO_SEASON = {1: "S", 2: "F", 3: "W", 4: "Sp"}


def chem_per_sample(chem_df, sample_cols, meta, round_to_season, site_aliases=None):
    """Reshape a site x {param}_{season} chem table into one row per
    microbiome sample, picking that sample's own season's columns. Sample
    columns look like '{round}-{no}-{sitecode+extra}'; the round-no prefix
    (e.g. '1-3b') is exactly metadata.csv's SampleID, which resolves the
    real Site/Round without having to parse the trailing site-code suffix.
    site_aliases covers sites whose ADP_6 code differs in this chem table
    (e.g. Chem_Se.Avg splits site MGY's effluent into MGYe1/MGYe2)."""
    site_aliases = site_aliases or {}
    rows = {}
    unmatched = []
    for col in sample_cols:
        sample_id = "-".join(col.split("-")[:2])
        if sample_id not in meta.index:
            unmatched.append((col, "no metadata row"))
            continue
        site = meta.loc[sample_id, "Site"]
        season = round_to_season[int(meta.loc[sample_id, "Round"])]
        lookup_site = site if site in chem_df.index else site_aliases.get(site, site)
        if lookup_site not in chem_df.index:
            unmatched.append((col, f"site {site} not in chem table"))
            continue
        suffix = f"_{season}"
        rows[col] = {c[: -len(suffix)]: chem_df.loc[lookup_site, c] for c in chem_df.columns if c.endswith(suffix)}
    return pd.DataFrame.from_dict(rows, orient="index"), unmatched


sample_cols_all = sorted(set(ARC_G.columns[1:]) | set(BAC_G.columns[1:]))
Sub_per_sample, sub_unmatched = chem_per_sample(Sub, sample_cols_all, meta, ROUND_TO_SEASON)
M_per_sample, m_unmatched = chem_per_sample(M, sample_cols_all, meta, ROUND_TO_SEASON, site_aliases={"MGY": "MGYe2"})
print(f"Sub_per_sample: {Sub_per_sample.shape}, unmatched samples: {len(sub_unmatched)}")
print(f"M_per_sample: {M_per_sample.shape}, unmatched samples: {len(m_unmatched)} {sorted(set(s for s, _ in m_unmatched))}")


def taxon_chem_long_corr(taxon_df, chem_per_sample_df, domain, chem_source):
    taxon_df = taxon_df.set_index(taxon_df.columns[0])
    common = [c for c in taxon_df.columns if c in chem_per_sample_df.index]
    taxa_t = taxon_df[common].T
    chem = chem_per_sample_df.loc[common]

    cross = pd.concat([taxa_t, chem], axis=1).corr(method="spearman").loc[taxa_t.columns, chem.columns]
    n_matrix = (~taxa_t.isna()).astype(int).to_numpy().T @ (~chem.isna()).astype(int).to_numpy()

    rows = []
    for i, taxon in enumerate(taxa_t.columns):
        for j, param in enumerate(chem.columns):
            rho, n = cross.iloc[i, j], n_matrix[i, j]
            if pd.isna(rho) or n < MIN_PAIRED_SAMPLES:
                continue
            rows.append((domain, chem_source, taxon, param, rho, abs(rho), n))
    return pd.DataFrame(rows, columns=["domain", "chem_source", "taxon", "parameter", "rho", "abs_rho", "n"])


corr_results = pd.concat(
    [
        taxon_chem_long_corr(ARC_G, Sub_per_sample, "ARC", "Sub"),
        taxon_chem_long_corr(ARC_G, M_per_sample, "ARC", "M"),
        taxon_chem_long_corr(BAC_G, Sub_per_sample, "BAC", "Sub"),
        taxon_chem_long_corr(BAC_G, M_per_sample, "BAC", "M"),
    ],
    ignore_index=True,
)

top30 = corr_results.sort_values("abs_rho", ascending=False).head(30).reset_index(drop=True)
print(f"\n{len(corr_results)} taxon-parameter pairs (n >= {MIN_PAIRED_SAMPLES}); top 30 by |rho|:")
top30


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

POSITIVE, NEGATIVE = "#2a78d6", "#e34948"
GRIDLINE, MUTED, PRIMARY, BASELINE = "#e1e0d9", "#898781", "#0b0b0b", "#c3c2b7"

plot_df = top30.sort_values("abs_rho", ascending=True).reset_index(drop=True)
labels = [f"{r.taxon} × {r.parameter}  ({r.domain}/{r.chem_source})" for r in plot_df.itertuples()]
colors = [POSITIVE if r.rho > 0 else NEGATIVE for r in plot_df.itertuples()]

fig, ax = plt.subplots(figsize=(9, 11))
ax.barh(labels, plot_df["rho"], color=colors, height=0.6, zorder=3)
ax.axvline(0, color=BASELINE, linewidth=1)
ax.set_xlim(-1, 1)
ax.set_xlabel("Spearman ρ", color=PRIMARY)
ax.set_title(f"Top {len(top30)} genus–chemistry correlations (|ρ|, n≥{MIN_PAIRED_SAMPLES})", color=PRIMARY, loc="left")
ax.tick_params(axis="y", labelsize=8, colors=PRIMARY)
ax.tick_params(axis="x", colors=MUTED)
ax.grid(axis="x", color=GRIDLINE, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(BASELINE)
ax.legend(handles=[Patch(color=POSITIVE, label="Positive"), Patch(color=NEGATIVE, label="Negative")], loc="lower right", frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

DOMAIN_TABLES = {"ARC": ARC_G, "BAC": BAC_G}
DIVERGING = LinearSegmentedColormap.from_list("diverging", [NEGATIVE, "#f0efec", POSITIVE])

top_genera = top30[["domain", "taxon"]].drop_duplicates().reset_index(drop=True)

abund_cols = {}
for domain, taxon in top_genera.itertuples(index=False):
    table = DOMAIN_TABLES[domain].set_index(DOMAIN_TABLES[domain].columns[0])
    label = taxon if taxon not in abund_cols else f"{taxon} ({domain})"
    abund_cols[label] = table.loc[taxon]

genus_abund = pd.DataFrame(abund_cols)
genus_corr = genus_abund.corr(method="spearman")

# hierarchical-cluster the order so correlated blocks sit together
dist = 1 - genus_corr.to_numpy()
np.fill_diagonal(dist, 0)
condensed = squareform((dist + dist.T) / 2, checks=False)
order = leaves_list(linkage(condensed, method="average"))
genus_corr = genus_corr.iloc[order, order]

fig, ax = plt.subplots(figsize=(9, 8))
mesh = ax.pcolormesh(genus_corr, cmap=DIVERGING, vmin=-1, vmax=1, edgecolors="white", linewidth=1.5)
ax.set_xticks(np.arange(len(genus_corr.columns)) + 0.5)
ax.set_yticks(np.arange(len(genus_corr.index)) + 0.5)
ax.set_xticklabels(genus_corr.columns, rotation=90, fontsize=8, color=PRIMARY)
ax.set_yticklabels(genus_corr.index, fontsize=8, color=PRIMARY)
ax.invert_yaxis()
ax.set_aspect("equal")
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Spearman ρ", color=PRIMARY)
cbar.ax.tick_params(colors=MUTED)
ax.set_title(f"Genus–genus abundance correlation — {len(genus_corr)} genera from the top {len(top30)} chemistry associations", color=PRIMARY, loc="left", fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import sys

from Bio import Phylo
from scipy.cluster.hierarchy import fcluster
from scipy.stats import rankdata

sys.path.insert(0, "../Phylospec")
from domain_merge_utils import fast_patristic  # reuses the repo's vectorised patristic-distance routine

# The heatmap's two blocks look cleanly separated by abundance correlation --
# test with ANOSIM (Clarke 1993) whether those SAME two clusters also
# separate on phylogenetic (patristic) distance, i.e. is the co-abundance
# block just shared ancestry, or something else (shared niche/function
# across unrelated lineages)? Distance is computed on each genus's ASV set
# via the domain's own ASV-level tree/taxonomy (same source as the Mantel-
# test cells above), as the average patristic distance between every ASV
# pair across the two genera -- clusters themselves are unchanged, still
# from the abundance-correlation clustering.
N_CLUSTERS = 2
ANOSIM_PERMUTATIONS = 999
ANOSIM_SEED = 0
QIIME_DIR = "qiimeresult"


def anosim(dist_matrix, groups, n_permutations, seed):
    """R in [-1, 1]: R~1 = groups fully separated (all within-group distances
    rank below all between-group ones), R~0 = no separation."""
    n = dist_matrix.shape[0]
    iu = np.triu_indices(n, k=1)
    ranks = rankdata(dist_matrix[iu])
    M = n * (n - 1) / 2

    def statistic(same_group_mask):
        r_within = ranks[same_group_mask].mean() if same_group_mask.any() else np.nan
        r_between = ranks[~same_group_mask].mean() if (~same_group_mask).any() else np.nan
        return (r_between - r_within) / (M / 2)

    R_obs = statistic(groups[iu[0]] == groups[iu[1]])
    rng = np.random.default_rng(seed)
    perm_R = np.empty(n_permutations)
    for i in range(n_permutations):
        perm_groups = rng.permutation(groups)
        perm_R[i] = statistic(perm_groups[iu[0]] == perm_groups[iu[1]])
    p_value = (np.sum(perm_R >= R_obs) + 1) / (n_permutations + 1)
    return R_obs, p_value


# cluster labels come from the abundance-correlation clustering (unchanged);
# same (pre-reorder) axis as `dist`/`condensed`
cluster_labels = fcluster(linkage(condensed, method="average"), N_CLUSTERS, criterion="maxclust")

domains_in_top = top_genera["domain"].unique()
assert len(domains_in_top) == 1, (
    f"genus-genus phylogenetic distance needs one shared tree; top30 spans domains "
    f"{list(domains_in_top)} -- would need the merged SEPP tree, not implemented here"
)
domain = domains_in_top[0]

tree = Phylo.read(f"{QIIME_DIR}/{domain}/rooted-tree.nwk", "newick")
tip_names = {t.name for t in tree.get_terminals()}
tax = pd.read_csv(f"{QIIME_DIR}/{domain}/metadata.tsv", sep="\t", skiprows=[1])  # ASV taxonomy, not sample metadata
tax["genus"] = tax["Taxon"].str.extract(r"g__([^;]*)")

labels, taxa = list(genus_abund.columns), list(top_genera["taxon"])
genus_asvs = {label: [a for a in tax.loc[tax["genus"] == taxon, "Feature ID"] if a in tip_names]
              for label, taxon in zip(labels, taxa)}
empty = [label for label, asvs in genus_asvs.items() if not asvs]
if empty:
    raise ValueError(f"no tree-matched ASVs found for: {empty}")

all_asvs = sorted({a for asvs in genus_asvs.values() for a in asvs})
D_asv = fast_patristic(tree, all_asvs)
asv_idx = {a: i for i, a in enumerate(all_asvs)}

n = len(labels)
dist_phylo = np.zeros((n, n))
for i in range(n):
    idx_i = [asv_idx[a] for a in genus_asvs[labels[i]]]
    for j in range(i + 1, n):
        idx_j = [asv_idx[a] for a in genus_asvs[labels[j]]]
        dist_phylo[i, j] = dist_phylo[j, i] = D_asv[np.ix_(idx_i, idx_j)].mean()

R_obs, p_value = anosim(dist_phylo, cluster_labels, ANOSIM_PERMUTATIONS, ANOSIM_SEED)

iu = np.triu_indices(len(cluster_labels), k=1)
same = cluster_labels[iu[0]] == cluster_labels[iu[1]]
within_phylo, between_phylo = dist_phylo[iu][same], dist_phylo[iu][~same]

sig = "significant" if p_value < 0.05 else "not significant"
print(f"cluster sizes: {list(np.bincount(cluster_labels)[1:])}")
print(f"mean within-group phylogenetic distance:  {within_phylo.mean():.4f}  (n={same.sum()} pairs)")
print(f"mean between-group phylogenetic distance: {between_phylo.mean():.4f}  (n={(~same).sum()} pairs)")
print(f"ANOSIM R={R_obs:.4f}, p={p_value:.4f} ({sig}, {ANOSIM_PERMUTATIONS} permutations)")


In [ ]:
import sys

from Bio import Phylo
from scipy.stats import spearmanr

sys.path.insert(0, "../Phylospec")
from domain_merge_utils import fast_patristic  # reuses the repo's vectorised patristic-distance routine

# Mantel test: is phylogenetic (patristic) distance between ASVs correlated
# with how dissimilar their relative-abundance profiles are across samples?
# Run separately per domain at ASV level (the tree's native resolution),
# using the same >=15% prevalence filter documented in README.md for
# ASV-level work, since the raw ASV count (~13,385 BAC / ~1,776 ARC) makes
# an all-pairs distance matrix intractable otherwise.
PREVALENCE_MIN = 0.15
N_PERMUTATIONS = 999
MANTEL_SEED = 0


def load_relabund(tsv_path):
    counts = pd.read_csv(tsv_path, sep="\t", skiprows=1, index_col=0)  # skip QIIME2's "# Constructed from biom file"
    return counts.div(counts.sum(axis=0), axis=1)


def prevalence_filter(df, min_prevalence):
    prevalence = (df > 0).mean(axis=1)
    return df.loc[prevalence >= min_prevalence]


def mantel_test(D_phylo, D_abund_dist, n_permutations, seed):
    """Spearman correlation between the two distance matrices' upper
    triangles, with a permutation test (row/column relabelling of one
    matrix) for significance -- the standard Mantel test."""
    n = D_phylo.shape[0]
    iu = np.triu_indices(n, k=1)
    v_phylo, v_abund = D_phylo[iu], D_abund_dist[iu]
    r_obs, _ = spearmanr(v_phylo, v_abund)

    rng = np.random.default_rng(seed)
    perm_r = np.empty(n_permutations)
    for i in range(n_permutations):
        perm = rng.permutation(n)
        perm_r[i] = spearmanr(v_phylo, D_abund_dist[np.ix_(perm, perm)][iu])[0]
    p_value = (np.sum(np.abs(perm_r) >= abs(r_obs)) + 1) / (n_permutations + 1)
    return r_obs, p_value, v_phylo, v_abund


def phylo_abundance_mantel(domain):
    relabund = prevalence_filter(load_relabund(f"qiimeresult/{domain}/table_filtered.tsv"), PREVALENCE_MIN)
    tree = Phylo.read(f"qiimeresult/{domain}/rooted-tree.nwk", "newick")
    asvs = list(relabund.index)

    D_phylo = fast_patristic(tree, asvs)
    abund_corr = relabund.T.corr(method="spearman").to_numpy()  # ASV x ASV, across samples
    D_abund_dist = 1 - abund_corr

    r_obs, p_value, v_phylo, v_abund_corr_dist = mantel_test(D_phylo, D_abund_dist, N_PERMUTATIONS, MANTEL_SEED)
    return dict(
        domain=domain, n_asvs=len(asvs), n_pairs=len(v_phylo),
        mantel_r=r_obs, p_value=p_value,
        v_phylo=v_phylo, v_abund_rho=1 - v_abund_corr_dist,
    )


mantel_results = {domain: phylo_abundance_mantel(domain) for domain in ["ARC", "BAC"]}
for domain, res in mantel_results.items():
    sig = "significant" if res["p_value"] < 0.05 else "not significant"
    print(f"{domain}: n_asvs={res['n_asvs']}, n_pairs={res['n_pairs']:,}, "
          f"Mantel r={res['mantel_r']:.4f}, p={res['p_value']:.4f} ({sig}, {N_PERMUTATIONS} permutations)")


In [ ]:
SEQ_BLUE = LinearSegmentedColormap.from_list("seqblue", ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, domain in zip(axes, ["ARC", "BAC"]):
    res = mantel_results[domain]
    hb = ax.hexbin(res["v_phylo"], res["v_abund_rho"], gridsize=35, cmap=SEQ_BLUE, mincnt=1)
    ax.set_xlabel("Phylogenetic (patristic) distance", color=PRIMARY)
    ax.set_ylabel("Abundance-profile Spearman ρ", color=PRIMARY)
    ax.set_title(f"{domain}  (n={res['n_asvs']} ASVs, {res['n_pairs']:,} pairs)", color=PRIMARY, loc="left")
    ax.axhline(0, color=BASELINE, linewidth=1)
    ax.grid(color=GRIDLINE, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    sig = "significant" if res["p_value"] < 0.05 else "not significant"
    ax.text(0.98, 0.03, f"Mantel r={res['mantel_r']:.3f}, p={res['p_value']:.3f} ({sig})",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color=PRIMARY)
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("pair count", color=PRIMARY)
    cb.ax.tick_params(colors=MUTED)

fig.suptitle("Phylogenetic distance vs. abundance-profile correlation, per ASV pair", color=PRIMARY, x=0.02, ha="left")
plt.tight_layout()
plt.show()


In [ ]:
# Same Mantel test, generalised across taxonomic ranks. Genus reuses the
# existing tree-aware collapse (output/genus_tree/genus_autorun/); Class/
# Order/Family were built the same way (reduce_features_for_tree_models.py
# --mode genus --rank {class,order,family}, added as a --rank option
# generalising its previously genus-hardcoded collapse -- default --rank
# genus reproduces the existing genus_autorun output byte-for-byte, verified
# by diffing table/tree/mapping/report against it) into
# output/genus_tree/taxlevel_autorun/{rank}/. Every rank's {domain}_table.csv
# columns are exactly that rank's {domain}_tree.nwk tip labels, so no extra
# filtering is needed at any level.
RANK_TREE_DIRS = {
    "Class": "../output/genus_tree/taxlevel_autorun/class",
    "Order": "../output/genus_tree/taxlevel_autorun/order",
    "Family": "../output/genus_tree/taxlevel_autorun/family",
    "Genus": "../output/genus_tree/genus_autorun",
}
RANK_ORDER = ["Class", "Order", "Family", "Genus"]  # shallow -> fine


def rank_phylo_abundance_mantel(domain, domain_lower, rank):
    tree_dir = RANK_TREE_DIRS[rank]
    table = pd.read_csv(f"{tree_dir}/{domain_lower}_table.csv", index_col="SampleID")
    relabund = table.div(table.sum(axis=1), axis=0)  # samples are rows here, so normalise row-wise
    tree = Phylo.read(f"{tree_dir}/{domain_lower}_tree.nwk", "newick")

    features = list(relabund.columns)
    D_phylo = fast_patristic(tree, features)
    abund_corr = relabund.corr(method="spearman").to_numpy()  # already feature x feature (samples are rows)
    D_abund_dist = 1 - abund_corr

    r_obs, p_value, v_phylo, v_abund_corr_dist = mantel_test(D_phylo, D_abund_dist, N_PERMUTATIONS, MANTEL_SEED)
    return dict(
        domain=domain, rank=rank, n_features=len(features), n_pairs=len(v_phylo),
        mantel_r=r_obs, p_value=p_value,
        v_phylo=v_phylo, v_abund_rho=1 - v_abund_corr_dist,
    )


rank_mantel_results = {
    (domain, rank): rank_phylo_abundance_mantel(domain, domain_lower, rank)
    for domain, domain_lower in [("ARC", "arc"), ("BAC", "bac")]
    for rank in RANK_ORDER
}

rank_mantel_table = pd.DataFrame(
    [{"domain": r["domain"], "rank": r["rank"], "n_features": r["n_features"], "n_pairs": r["n_pairs"],
      "mantel_r": r["mantel_r"], "p_value": r["p_value"], "significant": r["p_value"] < 0.05}
     for r in rank_mantel_results.values()]
)
rank_mantel_table["rank"] = pd.Categorical(rank_mantel_table["rank"], categories=RANK_ORDER, ordered=True)
rank_mantel_table = rank_mantel_table.sort_values(["domain", "rank"]).reset_index(drop=True)
rank_mantel_table


In [ ]:
def plot_mantel_results(ranks=RANK_ORDER):
    """Grouped ARC/BAC bar chart of Mantel r for the given rank(s), e.g.
    plot_mantel_results(["Genus"]) or plot_mantel_results(["Family", "Genus"])."""
    unknown = set(ranks) - set(RANK_ORDER)
    if unknown:
        raise ValueError(f"unknown rank(s) {sorted(unknown)}; choose from {RANK_ORDER}")
    ranks = [r for r in RANK_ORDER if r in ranks]  # keep canonical shallow->fine order

    fig, ax = plt.subplots(figsize=(max(7, 2 * len(ranks)), 5))
    width = 0.35
    x = np.arange(len(ranks))
    for offset, domain, color in [(-width / 2, "ARC", POSITIVE), (width / 2, "BAC", NEGATIVE)]:
        sub = rank_mantel_table[rank_mantel_table["domain"] == domain].set_index("rank").loc[ranks]
        ax.bar(x + offset, sub["mantel_r"], width, label=domain, color=color, zorder=3)
        for xi, (r_val, p_val) in zip(x + offset, zip(sub["mantel_r"], sub["p_value"])):
            marker = "*" if p_val < 0.05 else "ns"
            ax.text(xi, r_val + (0.01 if r_val >= 0 else -0.03), marker, ha="center",
                    fontsize=10 if marker == "*" else 7, color=PRIMARY)

    ax.axhline(0, color=BASELINE, linewidth=1)
    ax.set_xlim(-0.5, len(ranks) - 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(ranks, color=PRIMARY)
    ax.set_ylabel("Mantel r (Spearman, phylo distance vs. abundance dissimilarity)", color=PRIMARY)
    ax.set_title("Phylogenetic signal in abundance correlation, by taxonomic rank", color=PRIMARY, loc="left", fontsize=11)
    ax.grid(axis="y", color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.legend(frameon=False)
    ax.text(0.99, 0.02, "* p<0.05 (999 permutations)", transform=ax.transAxes, ha="right", fontsize=8, color=MUTED)

    plt.tight_layout()
    plt.show()


plot_mantel_results()  # all ranks by default; e.g. plot_mantel_results(["Genus"]) for just one


In [ ]:
def plot_mantel_hexbins(ranks=RANK_ORDER):
    """Phylo-distance vs. abundance-correlation hexbin, one row per rank,
    ARC/BAC columns -- the same panel as the ASV-level Mantel plot above,
    generalised across taxonomic ranks via rank_mantel_results. E.g.
    plot_mantel_hexbins(["Genus"]) or plot_mantel_hexbins(["Family", "Genus"])."""
    unknown = set(ranks) - set(RANK_ORDER)
    if unknown:
        raise ValueError(f"unknown rank(s) {sorted(unknown)}; choose from {RANK_ORDER}")
    ranks = [r for r in RANK_ORDER if r in ranks]

    fig, axes = plt.subplots(len(ranks), 2, figsize=(12, 5 * len(ranks)), squeeze=False)
    for row, rank in enumerate(ranks):
        for ax, domain in zip(axes[row], ["ARC", "BAC"]):
            res = rank_mantel_results[(domain, rank)]
            hb = ax.hexbin(res["v_phylo"], res["v_abund_rho"], gridsize=35, cmap=SEQ_BLUE, mincnt=1)
            ax.set_xlabel("Phylogenetic (patristic) distance", color=PRIMARY)
            ax.set_ylabel("Abundance-profile Spearman ρ", color=PRIMARY)
            ax.set_title(f"{domain} / {rank}  (n={res['n_features']} features, {res['n_pairs']:,} pairs)",
                         color=PRIMARY, loc="left")
            ax.axhline(0, color=BASELINE, linewidth=1)
            ax.grid(color=GRIDLINE, linewidth=0.6, zorder=0)
            ax.set_axisbelow(True)
            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)
            sig = "significant" if res["p_value"] < 0.05 else "not significant"
            ax.text(0.98, 0.03, f"Mantel r={res['mantel_r']:.3f}, p={res['p_value']:.3f} ({sig})",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color=PRIMARY)
            cb = fig.colorbar(hb, ax=ax)
            cb.set_label("pair count", color=PRIMARY)
            cb.ax.tick_params(colors=MUTED)

    fig.suptitle("Phylogenetic distance vs. abundance-profile correlation, per taxon pair", color=PRIMARY, x=0.02, ha="left")
    plt.tight_layout(rect=[0, 0, 1, 1 - 0.15 / len(ranks)])
    plt.show()


plot_mantel_hexbins()  # all ranks by default; e.g. plot_mantel_hexbins(["Genus"]) for just one


In [ ]:
# Redundancy screen via the variation log-ratio (VLR), a.k.a. proportionality
# (Lovell et al. 2015). For taxa j, k on a positive composition q:
#   v_jk = Var[log(q_j / q_k)]
# A small v_jk means the ratio is comparatively stable across samples --
# little independent log-ratio information -- which is a compositional-data-
# safe alternative to plain Pearson/Spearman correlation for flagging
# redundant features (correlation on relative abundances is inflated by
# closure; VLR is not). This does NOT prove interchangeable metabolism:
# co-absence and poorly measured low counts can create misleadingly low VLR,
# so pairs are only trusted where both taxa are adequately jointly detected.
#
# Runs on the same 21-genus table (`genus_abund`) behind the heatmap/ANOSIM
# cells above.

PSEUDOCOUNT_DEFAULT = genus_abund.to_numpy()[genus_abund.to_numpy() > 0].min() / 2  # documented zero handling:
# additive pseudocount = half the smallest observed nonzero relative abundance in this table
MIN_JOINT_DETECTION = 0.5  # require >=50% of samples to have BOTH taxa nonzero before trusting a low VLR

taxa = list(genus_abund.columns)
n_taxa = len(taxa)
detected = genus_abund.to_numpy() > 0


def vlr_from_logq(logq_arr):
    out = np.zeros((n_taxa, n_taxa))
    for i in range(n_taxa):
        for j in range(i + 1, n_taxa):
            out[i, j] = out[j, i] = (logq_arr[:, i] - logq_arr[:, j]).var(ddof=1)
    return out


def zero_handled_log(pseudocount):
    q = genus_abund.where(genus_abund > 0, pseudocount)
    return np.log(q.to_numpy())


logq = zero_handled_log(PSEUDOCOUNT_DEFAULT)
vlr_pooled = vlr_from_logq(logq)

iu = np.triu_indices(n_taxa, k=1)
rows = []
for i, j in zip(*iu):
    joint_det = (detected[:, i] & detected[:, j]).mean()
    rows.append((taxa[i], taxa[j], vlr_pooled[i, j], joint_det))
vlr_df = pd.DataFrame(rows, columns=["taxon_j", "taxon_k", "vlr", "joint_detection"])
vlr_df["adequate_detection"] = vlr_df["joint_detection"] >= MIN_JOINT_DETECTION

# -- zero-handling sensitivity check: does a very different pseudocount change the ranking? --
PSEUDOCOUNT_ALT = PSEUDOCOUNT_DEFAULT * 10
vlr_alt = vlr_from_logq(zero_handled_log(PSEUDOCOUNT_ALT))
vlr_df["vlr_alt_pseudocount"] = vlr_alt[iu]
rank_corr_zero, _ = spearmanr(vlr_df["vlr"], vlr_df["vlr_alt_pseudocount"])

# -- pooled vs. within-site: removes between-site variation, isolating seasonal
# (within-site) covariation. EXPLORATORY ONLY -- most sites have just 4 visits. --
sample_ids = ["-".join(c.split("-")[:2]) for c in genus_abund.index]
site_of_sample = pd.Series(
    [meta.loc[sid, "Site"] if sid in meta.index else np.nan for sid in sample_ids],
    index=genus_abund.index, name="Site",
)
logq_df = pd.DataFrame(logq, index=genus_abund.index, columns=taxa)
logq_within = (logq_df - logq_df.groupby(site_of_sample).transform("mean")).to_numpy()
vlr_within = vlr_from_logq(logq_within)
vlr_df["vlr_within_site"] = vlr_within[iu]
rank_corr_site, _ = spearmanr(vlr_df["vlr"], vlr_df["vlr_within_site"])

adequate = vlr_df[vlr_df["adequate_detection"]].sort_values("vlr").reset_index(drop=True)
print(f"pseudocount (half min nonzero) = {PSEUDOCOUNT_DEFAULT:.4f}")
print(f"{len(vlr_df)} pairs; {vlr_df['adequate_detection'].sum()} meet joint detection >= {MIN_JOINT_DETECTION}")
print(f"zero-handling sensitivity (pseudocount x10): VLR rank correlation = {rank_corr_zero:.3f}")
print(f"pooled vs. within-site VLR rank correlation = {rank_corr_site:.3f} "
      f"(within-site is exploratory: sites have {site_of_sample.value_counts().median():.0f} visits typically)")
print("\nMost redundant (lowest-VLR) pairs among adequately-detected pairs:")
adequate.head(15)


In [ ]:
import re

# Sensitivity to major operating conditions: is a "stable" (low-VLR) log-ratio
# actually just two taxa independently tracking the same digester temperature
# or feedstock, rather than tracking each other? Regress each top pair's log-
# ratio on T_C (parsed from strings like "35~38"/"35±2") and feedstock
# (substrate_type, rare categories folded into "Other") and report R^2 -- a
# high R^2 means shared operating conditions explain much of the "stability."


def parse_temp(s):
    s = str(s)
    if "±" in s:  # "35±2" -> center value, not the tolerance
        return float(s.split("±")[0])
    nums = re.findall(r"\d+\.?\d*", s)  # "25-37", "25~34" -> mean of endpoints
    return np.mean([float(x) for x in nums])


T_numeric = meta["T_C"].apply(parse_temp)
top_substrates = meta["substrate_type"].value_counts().head(5).index
substrate_bucket = meta["substrate_type"].where(meta["substrate_type"].isin(top_substrates), "Other")


def covariate_design(sample_cols):
    sample_ids = ["-".join(c.split("-")[:2]) for c in sample_cols]
    t = T_numeric.reindex(sample_ids).to_numpy()
    dummies = pd.get_dummies(substrate_bucket.reindex(sample_ids), drop_first=True).to_numpy(dtype=float)
    return np.column_stack([np.ones(len(sample_cols)), t, dummies])


def r2_explained(y, X):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    ss_res, ss_tot = (resid ** 2).sum(), ((y - y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot


X_design = covariate_design(genus_abund.index)
covariate_r2 = []
for _, row in adequate.head(10).iterrows():
    j, k = taxa.index(row["taxon_j"]), taxa.index(row["taxon_k"])
    log_ratio = logq[:, j] - logq[:, k]
    covariate_r2.append(r2_explained(log_ratio, X_design))

adequate_with_r2 = adequate.head(10).copy()
adequate_with_r2["r2_temperature_feedstock"] = covariate_r2
print("R^2 of log(taxon_j/taxon_k) ~ temperature + feedstock, for the top redundant pairs:")
print("(low R^2 = the stable ratio is not just a shared response to these operating conditions)")
adequate_with_r2[["taxon_j", "taxon_k", "vlr", "joint_detection", "r2_temperature_feedstock"]]


In [ ]:
from scipy.stats import t as t_dist
from statsmodels.stats.multitest import multipletests

# The top-30 chemistry screen (cell above) was selected from 9,724 comparisons
# with unequal paired sample sizes per pair -- neither fact is visible in a
# bare |rho| ranking. Add per-pair p-values (analytic t-approximation, the
# same asymptotic method scipy.stats.spearmanr uses without ties/permutation)
# and Benjamini-Hochberg FDR control across all 9,724 tests, so "top 30 by
# |rho|" claims can be checked against "significant after multiple-testing
# correction," with n reported alongside every claim.

rho_all = corr_results["rho"].to_numpy()
n_all = corr_results["n"].to_numpy()
with np.errstate(divide="ignore", invalid="ignore"):
    t_stat = rho_all * np.sqrt((n_all - 2) / (1 - rho_all ** 2))
p_analytic = 2 * t_dist.sf(np.abs(t_stat), df=n_all - 2)

corr_results_fdr = corr_results.copy()
corr_results_fdr["p_value"] = p_analytic
_, corr_results_fdr["q_value"], _, _ = multipletests(p_analytic, method="fdr_bh")

top30_fdr = corr_results_fdr.sort_values("abs_rho", ascending=False).head(30).reset_index(drop=True)
print(f"{len(corr_results_fdr)} tests; {(corr_results_fdr['q_value'] < 0.05).sum()} significant at BH q<0.05")
print(f"of the top-30 by |rho|: {(top30_fdr['q_value'] < 0.05).sum()}/30 remain significant at BH q<0.05")
print(f"paired sample counts among the top 30 range n={top30_fdr['n'].min()}-{top30_fdr['n'].max()} "
      "(report n alongside any individual association claim -- pairs were NOT all tested on the same n)")
print()
print("CAVEAT: this screen was optimized to find genera associated with CHEMISTRY parameters.")
print("It is not a feature-selection result for any particular prediction target (e.g. the 5")
print("warning flags) -- do not restrict downstream model inputs to these genera on this basis.")

top30_fdr[["domain", "chem_source", "taxon", "parameter", "rho", "n", "p_value", "q_value"]]


In [ ]:
# Site-block bootstrap: resample whole SITES with replacement (keeping each
# selected site's own seasonal observations together), recompute the
# relationships, and see how much they move. This is a STABILITY check on
# repeated-measures data (up to 4 seasons/site are not independent draws) --
# distinct from the taxon-label permutations the Mantel tests above use for
# SIGNIFICANCE (those shuffle which taxon each distance-matrix row belongs
# to; this instead resamples which observational units -- sites -- appear in
# the data at all).
N_SITE_BOOT = 300
SITE_BOOT_SEED = 0

# -- (1) stability of the top redundant (lowest-VLR) pairs --
unique_sites = site_of_sample.dropna().unique()
top_pairs_for_boot = list(zip(adequate.head(10)["taxon_j"], adequate.head(10)["taxon_k"]))

rng = np.random.default_rng(SITE_BOOT_SEED)
boot_vlr = {pair: np.empty(N_SITE_BOOT) for pair in top_pairs_for_boot}
for b in range(N_SITE_BOOT):
    chosen_sites = rng.choice(unique_sites, size=len(unique_sites), replace=True)
    site_counts = pd.Series(chosen_sites).value_counts()
    rep_index = [c for c in genus_abund.index for _ in range(site_counts.get(site_of_sample[c], 0))]
    logq_boot = zero_handled_log(PSEUDOCOUNT_DEFAULT)[[genus_abund.index.get_loc(c) for c in rep_index]]
    for tj, tk in top_pairs_for_boot:
        j, k = taxa.index(tj), taxa.index(tk)
        boot_vlr[(tj, tk)][b] = (logq_boot[:, j] - logq_boot[:, k]).var(ddof=1)

print(f"Site-block bootstrap (n={N_SITE_BOOT}) 95% CI for the top redundant pairs' VLR:")
for (tj, tk), vals in boot_vlr.items():
    point = adequate.set_index(["taxon_j", "taxon_k"]).loc[(tj, tk), "vlr"]
    lo, hi = np.percentile(vals, [2.5, 97.5])
    print(f"  {tj} vs {tk}: point={point:.2f}, 95% CI=[{lo:.2f}, {hi:.2f}]")

# -- (2) stability of the ARC-minus-BAC Mantel effect --
def build_site_index(sample_cols, meta_df):
    out = {}
    for col in sample_cols:
        sample_id = "-".join(col.split("-")[:2])
        if sample_id in meta_df.index:
            out[col] = meta_df.loc[sample_id, "Site"]
    return out


def site_to_cols_map(sample_site):
    out = {}
    for col, site in sample_site.items():
        out.setdefault(site, []).append(col)
    return out


def domain_mantel_inputs(domain):
    relabund_d = prevalence_filter(load_relabund(f"qiimeresult/{domain}/table_filtered.tsv"), PREVALENCE_MIN)
    tree_d = Phylo.read(f"qiimeresult/{domain}/rooted-tree.nwk", "newick")
    D_phylo_d = fast_patristic(tree_d, list(relabund_d.index))
    return relabund_d, D_phylo_d, site_to_cols_map(build_site_index(list(relabund_d.columns), meta))


def resampled_mantel_r(relabund_d, D_phylo_d, site2cols_d, chosen_sites):
    cols = [c for s in chosen_sites for c in site2cols_d.get(s, [])]
    if len(cols) < 10:
        return np.nan
    abund_corr = relabund_d[cols].T.corr(method="spearman").to_numpy()
    iu_d = np.triu_indices(abund_corr.shape[0], k=1)
    return spearmanr(D_phylo_d[iu_d], (1 - abund_corr)[iu_d])[0]


arc_relabund, arc_Dphylo, arc_site2cols = domain_mantel_inputs("ARC")
bac_relabund, bac_Dphylo, bac_site2cols = domain_mantel_inputs("BAC")
all_sites_mantel = sorted(set(arc_site2cols) | set(bac_site2cols))

rng2 = np.random.default_rng(SITE_BOOT_SEED)
boot_r_arc = np.empty(N_SITE_BOOT)
boot_r_bac = np.empty(N_SITE_BOOT)
for b in range(N_SITE_BOOT):
    chosen = rng2.choice(all_sites_mantel, size=len(all_sites_mantel), replace=True)
    boot_r_arc[b] = resampled_mantel_r(arc_relabund, arc_Dphylo, arc_site2cols, chosen)
    boot_r_bac[b] = resampled_mantel_r(bac_relabund, bac_Dphylo, bac_site2cols, chosen)

valid = ~(np.isnan(boot_r_arc) | np.isnan(boot_r_bac))
diff = boot_r_arc[valid] - boot_r_bac[valid]

print(f"\nARC-minus-BAC Mantel effect, site-block bootstrap (n={valid.sum()} valid of {N_SITE_BOOT}):")
print(f"  ARC r: point={mantel_results['ARC']['mantel_r']:.3f}, "
      f"95% CI=[{np.percentile(boot_r_arc[valid], 2.5):.3f}, {np.percentile(boot_r_arc[valid], 97.5):.3f}]")
print(f"  BAC r: point={mantel_results['BAC']['mantel_r']:.3f}, "
      f"95% CI=[{np.percentile(boot_r_bac[valid], 2.5):.3f}, {np.percentile(boot_r_bac[valid], 97.5):.3f}]")
print(f"  ARC-BAC diff: point={mantel_results['ARC']['mantel_r'] - mantel_results['BAC']['mantel_r']:.3f}, "
      f"95% CI=[{np.percentile(diff, 2.5):.3f}, {np.percentile(diff, 97.5):.3f}]")
print(f"  fraction of site resamples with ARC > BAC: {(diff > 0).mean():.3f}")
